# Song Genre Classifier — XGBoost, AdaBoost, and Lyrics+Audio Only

This cleaned notebook keeps only the requested classifiers:

1. **XGBoost** boosted-tree classifiers
2. **AdaBoost** boosted-tree classifiers
3. **Lyrics + audio neural classifier**

All comparison/evaluation cells use the same shared train/validation/test split so the results are comparable.


In [3]:
import os
import re
import warnings
warnings.filterwarnings('ignore')
import ast
import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy import sparse

from sklearn.preprocessing import MultiLabelBinarizer, StandardScaler
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.multiclass import OneVsRestClassifier
from sklearn.ensemble import AdaBoostClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import (
    f1_score, hamming_loss, precision_score, recall_score,
    classification_report
)
from sklearn.model_selection import train_test_split

try:
    from xgboost import XGBClassifier
    HAS_XGBOOST = True
except Exception:
    HAS_XGBOOST = False
    XGBClassifier = None

try:
    from iterstrat.ml_stratifiers import MultilabelStratifiedShuffleSplit
    HAS_ITERSTRAT = True
except Exception:
    HAS_ITERSTRAT = False

import tensorflow as tf
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences
from tensorflow.keras.models import Model
from tensorflow.keras.layers import (
    Input, Embedding, GlobalAveragePooling1D, Dense, Dropout,
    BatchNormalization, Concatenate
)
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau
from tensorflow.keras.regularizers import l2
from tensorflow.keras.metrics import Precision, Recall

SEED = 42
np.random.seed(SEED)
tf.random.set_seed(SEED)

print('Iterative stratification available:', HAS_ITERSTRAT)
print('XGBoost available:', HAS_XGBOOST)


Iterative stratification available: True
XGBoost available: True


## 1) Load CSV and normalize columns


In [4]:
# Add your local path here if needed.
CANDIDATE_PATHS = [
    '/Users/tukermoose/Comp-SP26/Comp_329_GP/songs.csv'
]

CSV_PATH = next((p for p in CANDIDATE_PATHS if os.path.exists(p)), None)
if CSV_PATH is None:
    raise FileNotFoundError(f'Could not find the CSV. Checked: {CANDIDATE_PATHS}')

print('Using CSV:', CSV_PATH)

df = pd.read_csv(
    CSV_PATH,
    sep=None,
    engine='python',
    quotechar='"',
    escapechar='`',
    on_bad_lines='warn'
)

print('Raw shape before normalization:', df.shape)
print('Original columns:', list(df.columns))

# Normalize common alternate names.
# IMPORTANT:
# - The CSV uses `genre`, so rename it to `track_genre`.
# - The CSV uses `niche_genres` plural, which contains stringified lists like:
#   '["groove metal", "metal"]'.
# - Do NOT create/use fake `niche_genre = "unknown"` when `niche_genres` exists.
rename_map = {
    'id': 'track_id',
    'name': 'track_name',
    'genre': 'track_genre',
}
df = df.rename(columns=rename_map)

# If a dataset has singular niche_genre but not plural niche_genres, normalize it.
if 'niche_genres' not in df.columns and 'niche_genre' in df.columns:
    df = df.rename(columns={'niche_genre': 'niche_genres'})

# Required defaults.
required_defaults = {
    'track_id': '',
    'track_name': '',
    'artists': '',
    'album_name': '',
    'lyrics': '',
    'track_genre': 'unknown',
    'niche_genres': '[]',
    'danceability': np.nan,
    'energy': np.nan,
    'key': np.nan,
    'loudness': np.nan,
    'mode': np.nan,
    'speechiness': np.nan,
    'acousticness': np.nan,
    'instrumentalness': np.nan,
    'liveness': np.nan,
    'valence': np.nan,
    'tempo': np.nan,
    'time_signature': 4,
    'duration_ms': np.nan,
    'explicit': 0,
    'popularity': 0,
}

for col, default in required_defaults.items():
    if col not in df.columns:
        df[col] = default

numeric_cols = [
    'danceability', 'energy', 'key', 'loudness', 'mode', 'speechiness',
    'acousticness', 'instrumentalness', 'liveness', 'valence', 'tempo',
    'time_signature', 'duration_ms', 'explicit', 'popularity'
]

for col in numeric_cols:
    df[col] = pd.to_numeric(df[col], errors='coerce')

print('Shape after normalization:', df.shape)
print('Final columns:', list(df.columns))


Using CSV: /Users/tukermoose/Comp-SP26/Comp_329_GP/songs.csv
Raw shape before normalization: (550622, 24)
Original columns: ['id', 'name', 'album_name', 'artists', 'danceability', 'energy', 'key', 'loudness', 'mode', 'speechiness', 'acousticness', 'instrumentalness', 'liveness', 'valence', 'tempo', 'duration_ms', 'lyrics', 'year', 'genre', 'popularity', 'total_artist_followers', 'avg_artist_popularity', 'artist_ids', 'niche_genres']
Shape after normalization: (550622, 26)
Final columns: ['track_id', 'track_name', 'album_name', 'artists', 'danceability', 'energy', 'key', 'loudness', 'mode', 'speechiness', 'acousticness', 'instrumentalness', 'liveness', 'valence', 'tempo', 'duration_ms', 'lyrics', 'year', 'track_genre', 'popularity', 'total_artist_followers', 'avg_artist_popularity', 'artist_ids', 'niche_genres', 'time_signature', 'explicit']


## 2) Find all unique genre and niche_genre labels before grouping


- Uses the real `niche_genres` column from `songs.csv` instead of the fake default `niche_genre = 'unknown'` column.
- Parses stringified niche lists such as `['groove metal', 'metal']` into actual Python lists.
- Removes numeric `track_genre` labels such as `0`, `4`, `10`, `11`, and `17` before modeling.
- Groups duplicate tracks using parsed `niche_genres`, so `genres` and `niche_genres` both become real multilabel targets.
- Keeps one shared split so AdaBoost, XGBoost, and the multi-head lyrics+audio model compare the exact same test songs.

In [5]:
BAD_LABELS = {'', 'unknown', 'nan', 'none', 'null', '[]'}

def clean_label(value):
    if pd.isna(value):
        return ''
    return str(value).strip()


def is_valid_label(value):
    label = clean_label(value)
    return (
        label.lower() not in BAD_LABELS
        and not label.isdigit()
    )


def parse_niche_list(value):
    """
    Converts a niche genre value into a clean Python list.

    Handles:
    - '["groove metal", "metal"]'
    - "['groove metal', 'metal']"
    - 'groove metal, metal'
    - a Python list if it is already parsed
    """
    if isinstance(value, list):
        raw_items = value
    else:
        value = clean_label(value)

        if not value or value.lower() in BAD_LABELS:
            return []

        raw_items = None

        # First try literal_eval/json-style parsing for stringified lists.
        try:
            parsed = ast.literal_eval(value)
            if isinstance(parsed, list):
                raw_items = parsed
        except Exception:
            raw_items = None

        # Fallback for comma-separated strings.
        if raw_items is None:
            if ',' in value:
                raw_items = value.split(',')
            else:
                raw_items = [value]

    cleaned = []
    for item in raw_items:
        label = clean_label(item).strip('"').strip("'").strip()
        if is_valid_label(label):
            cleaned.append(label)

    return sorted(set(cleaned))


# --- CLEAN ORIGINAL LABEL COLUMNS ---
df['track_genre'] = df['track_genre'].apply(clean_label)
df['niche_genres'] = df['niche_genres'].apply(clean_label)

# --- PARSE NICHE GENRES INTO ACTUAL LISTS ---
df['niche_genres_parsed'] = df['niche_genres'].apply(parse_niche_list)

# --- UNIQUE TRACK GENRES ---
unique_track_genres_raw = sorted(set(g for g in df['track_genre'] if clean_label(g)))
unique_track_genres = sorted(set(g for g in df['track_genre'] if is_valid_label(g)))

# --- UNIQUE NICHE GENRES, FLATTENED FROM LISTS ---
unique_niche_genres = sorted(set(
    genre
    for row in df['niche_genres_parsed']
    for genre in row
    if is_valid_label(genre)
))

numeric_track_genres = [g for g in unique_track_genres_raw if clean_label(g).isdigit()]

print("Raw track_genre labels:")
print(unique_track_genres_raw)
print("Raw track_genre count:", len(unique_track_genres_raw))

print("\nClean track_genre labels:")
print(unique_track_genres)
print("Clean track_genre count:", len(unique_track_genres))

print("\nNumeric track_genre labels removed:")
print(numeric_track_genres)

print("\nUnique niche_genres labels parsed from list column:")
print(unique_niche_genres)
print("niche_genres count:", len(unique_niche_genres))

if len(unique_niche_genres) == 0:
    raise ValueError(
        "No usable niche genres were found. Check that the CSV column is named "
        "`niche_genres` and contains values like '[\"groove metal\", \"metal\"]'."
    )


Raw track_genre labels:
['0', '10', '11', '17', '4', 'Blues', 'Classical', 'Country', 'Electronic', 'Folk', 'Hip-Hop', 'Jazz', 'Pop', 'R&B', 'Rock']
Raw track_genre count: 15

Clean track_genre labels:
['Blues', 'Classical', 'Country', 'Electronic', 'Folk', 'Hip-Hop', 'Jazz', 'Pop', 'R&B', 'Rock']
Clean track_genre count: 10

Numeric track_genre labels removed:
['0', '10', '11', '17', '4']

Unique niche_genres labels parsed from list column:
['3 step', 'acid house', 'acid jazz', 'acid rock', 'acid techno', 'acoustic country', 'acoustic pop', 'adult standards', 'afrikaans pop', 'afro adura', 'afro house', 'afro r&b', 'afro soul', 'afro tech', 'afro-cuban jazz', 'afrobeat', 'afrobeats', 'afrogospel', 'afropiano', 'afropop', 'afroswing', 'agronejo', 'algerian chaabi', 'alt country', 'alternative dance', 'alternative hip hop', 'alternative metal', 'alternative pop', 'alternative r&b', 'alternative rock', 'alté', 'amapiano', 'ambient', 'ambient folk', 'ambient jazz', 'americana', 'anatolian

## 3) Clean lyrics and remove invalid numeric labels


In [6]:
def clean_lyrics(text: str) -> str:
    if not isinstance(text, str):
        return ''

    text = text.replace('\n', ' ').replace('\r', ' ')
    text = re.sub(r'\[.*?\]', ' ', text)
    text = re.sub(r"[^a-zA-Z0-9'\s]", ' ', text)
    text = re.sub(r'\s+', ' ', text).strip().lower()
    return text


def first_nonempty(series):
    for value in series:
        if isinstance(value, str) and value.strip():
            return value
    return ''


def clean_label_list(series):
    labels = []
    for value in series.dropna():
        label = clean_label(value)
        if is_valid_label(label):
            labels.append(label)
    return sorted(set(labels))


def flatten_label_lists(series):
    labels = []
    for value in series.dropna():
        if isinstance(value, list):
            labels.extend(value)
        else:
            labels.extend(parse_niche_list(value))
    return sorted(set(label for label in labels if is_valid_label(label)))


df['lyrics_clean'] = df['lyrics'].apply(clean_lyrics)

# Keep rows with:
# 1) a valid non-numeric genre
# 2) at least one parsed niche genre
# 3) usable lyrics and track_id
valid_track_mask = df['track_genre'].apply(is_valid_label)
valid_niche_mask = df['niche_genres_parsed'].apply(lambda labels: len(labels) > 0)
valid_text_mask = df['lyrics_clean'].astype(str).str.strip().ne('')
valid_id_mask = df['track_id'].astype(str).str.strip().ne('')

df = df[valid_track_mask & valid_niche_mask & valid_text_mask & valid_id_mask].copy()

print('Shape after removing invalid labels / missing lyrics:', df.shape)
print('Remaining unique track genres:', sorted(df['track_genre'].unique()))
print('Remaining unique niche labels:', sorted({g for labels in df['niche_genres_parsed'] for g in labels})[:50])
print('Remaining unique niche label count:', len(sorted({g for labels in df['niche_genres_parsed'] for g in labels})))


Shape after removing invalid labels / missing lyrics: (544460, 28)
Remaining unique track genres: ['Blues', 'Classical', 'Country', 'Electronic', 'Folk', 'Hip-Hop', 'Jazz', 'Pop', 'R&B', 'Rock']
Remaining unique niche labels: ['3 step', 'acid house', 'acid jazz', 'acid rock', 'acid techno', 'acoustic country', 'acoustic pop', 'adult standards', 'afrikaans pop', 'afro adura', 'afro house', 'afro r&b', 'afro soul', 'afro tech', 'afro-cuban jazz', 'afrobeat', 'afrobeats', 'afrogospel', 'afropiano', 'afropop', 'afroswing', 'agronejo', 'algerian chaabi', 'alt country', 'alternative dance', 'alternative hip hop', 'alternative metal', 'alternative pop', 'alternative r&b', 'alternative rock', 'alté', 'amapiano', 'ambient', 'ambient folk', 'ambient jazz', 'americana', 'anatolian rock', 'anime', 'anime rap', 'anti-folk', 'aor', 'arabesk', 'arabic hip hop', 'arena rock', 'argentine rock', 'argentine trap', 'arrocha', 'art pop', 'art rock', 'asakaa']
Remaining unique niche label count: 754


## 4) Merge duplicate songs into one song-level dataframe


In [7]:
song_df = (
    df.groupby('track_id', as_index=False)
      .agg({
          'track_name': 'first',
          'artists': 'first',
          'album_name': 'first',
          'lyrics_clean': first_nonempty,
          'track_genre': clean_label_list,
          'niche_genres_parsed': flatten_label_lists,
          'danceability': 'first',
          'energy': 'first',
          'key': 'first',
          'loudness': 'first',
          'mode': 'first',
          'speechiness': 'first',
          'acousticness': 'first',
          'instrumentalness': 'first',
          'liveness': 'first',
          'valence': 'first',
          'tempo': 'first',
          'time_signature': 'first',
          'duration_ms': 'first',
          'explicit': 'first',
          'popularity': 'first',
      })
      .rename(columns={
          'lyrics_clean': 'lyrics',
          'track_genre': 'genres',
          'niche_genres_parsed': 'niche_genres',
      })
)

# Drop rows that still do not have usable ids, lyrics, or labels.
song_df = song_df[song_df['track_id'].astype(str).str.strip() != ''].copy()
song_df = song_df[song_df['lyrics'].astype(str).str.strip() != ''].copy()
song_df = song_df[
    (song_df['genres'].apply(len) > 0) &
    (song_df['niche_genres'].apply(len) > 0)
].copy()

print('Song-level shape:', song_df.shape)
display(song_df[['track_id', 'track_name', 'genres', 'niche_genres']].head(5))


Song-level shape: (544460, 22)


,track_id,track_name,genres,niche_genres
0,0001Lyv0YTjkZSqzT4WkLy,Eye Of The Hurricane,[Rock],"[noise rock, post-punk, proto-punk]"
1,0001piYJu94Ec4hJFytG5G,Nothing but a Shade,[Rock],"[gothic metal, symphonic metal]"
2,0005VnpISGYLSGrXg9TEJS,Space Loneliness,[Rock],"[gothic metal, gothic rock, industrial metal, ..."
3,000B6fUCVkSThxEbewDZ8r,Fight or Flight,[Rock],"[heavy metal, power metal, speed metal]"
4,000CC8EParg64OmTxVnZ0p,It's All Coming Back To Me Now - Cover of Céli...,[Pop],[christmas]


## 5) Confirm unique final labels


In [8]:
unique_genres = sorted({g for labels in song_df['genres'] for g in labels})
unique_niche_genres = sorted({g for labels in song_df['niche_genres'] for g in labels})

print('Final unique genre labels:')
print(unique_genres)
print('Number of genre labels:', len(unique_genres))

print('\nFinal unique niche_genre labels:')
print(unique_niche_genres)
print('Number of niche_genre labels:', len(unique_niche_genres))


Final unique genre labels:
['Blues', 'Classical', 'Country', 'Electronic', 'Folk', 'Hip-Hop', 'Jazz', 'Pop', 'R&B', 'Rock']
Number of genre labels: 10

Final unique niche_genre labels:
['3 step', 'acid house', 'acid jazz', 'acid rock', 'acid techno', 'acoustic country', 'acoustic pop', 'adult standards', 'afrikaans pop', 'afro adura', 'afro house', 'afro r&b', 'afro soul', 'afro tech', 'afro-cuban jazz', 'afrobeat', 'afrobeats', 'afrogospel', 'afropiano', 'afropop', 'afroswing', 'agronejo', 'algerian chaabi', 'alt country', 'alternative dance', 'alternative hip hop', 'alternative metal', 'alternative pop', 'alternative r&b', 'alternative rock', 'alté', 'amapiano', 'ambient', 'ambient folk', 'ambient jazz', 'americana', 'anatolian rock', 'anime', 'anime rap', 'anti-folk', 'aor', 'arabesk', 'arabic hip hop', 'arena rock', 'argentine rock', 'argentine trap', 'arrocha', 'art pop', 'art rock', 'asakaa', 'aussie drill', 'avant-garde', 'axé', 'azonto', 'bacardi', 'bachata', 'ballet', 'ballroo

## 6) Build two multilabel targets


In [9]:
mlb_genre = MultiLabelBinarizer()
Y_genre = mlb_genre.fit_transform(song_df['genres'])

mlb_niche = MultiLabelBinarizer()
Y_niche = mlb_niche.fit_transform(song_df['niche_genres'])

print('Genre target shape:', Y_genre.shape)
print('Genre classes:', list(mlb_genre.classes_))

print('\nNiche target shape:', Y_niche.shape)
print('Niche classes:', list(mlb_niche.classes_))


Genre target shape: (544460, 10)
Genre classes: ['Blues', 'Classical', 'Country', 'Electronic', 'Folk', 'Hip-Hop', 'Jazz', 'Pop', 'R&B', 'Rock']

Niche target shape: (544460, 754)
Niche classes: ['3 step', 'acid house', 'acid jazz', 'acid rock', 'acid techno', 'acoustic country', 'acoustic pop', 'adult standards', 'afrikaans pop', 'afro adura', 'afro house', 'afro r&b', 'afro soul', 'afro tech', 'afro-cuban jazz', 'afrobeat', 'afrobeats', 'afrogospel', 'afropiano', 'afropop', 'afroswing', 'agronejo', 'algerian chaabi', 'alt country', 'alternative dance', 'alternative hip hop', 'alternative metal', 'alternative pop', 'alternative r&b', 'alternative rock', 'alté', 'amapiano', 'ambient', 'ambient folk', 'ambient jazz', 'americana', 'anatolian rock', 'anime', 'anime rap', 'anti-folk', 'aor', 'arabesk', 'arabic hip hop', 'arena rock', 'argentine rock', 'argentine trap', 'arrocha', 'art pop', 'art rock', 'asakaa', 'aussie drill', 'avant-garde', 'axé', 'azonto', 'bacardi', 'bachata', 'ballet'

## 7) One shared train/validation/test split used by AdaBoost, XGBoost, and the multi-head model

In [10]:
# --- SHARED INPUTS ---
# These arrays are created once and then split once.
# AdaBoost, XGBoost, and the multi-head lyrics+audio model all use the same split dictionary below.
X_text = song_df['lyrics'].astype(str).values
meta_titles = song_df['track_name'].astype(str).values
meta_artists = song_df['artists'].astype(str).values

AUDIO_FEATURES = [
    'danceability', 'energy', 'key', 'loudness', 'mode', 'speechiness',
    'acousticness', 'instrumentalness', 'liveness', 'valence', 'tempo',
    'time_signature', 'duration_ms', 'explicit', 'popularity'
]

X_audio = song_df[AUDIO_FEATURES].copy()
X_audio['explicit'] = pd.to_numeric(X_audio['explicit'], errors='coerce')
X_audio = X_audio.fillna(X_audio.median(numeric_only=True))
X_audio = X_audio.astype(float).values

# Safety checks: both label matrices must describe the same song rows.
assert len(X_text) == X_audio.shape[0] == Y_genre.shape[0] == Y_niche.shape[0], "Input/label row mismatch."

# Use combined genre+niche labels only for splitting.
# This makes the same songs appear in train/validation/test for every model.
Y_joint_for_split = np.hstack([Y_genre, Y_niche])


def multilabel_dual_split(X_text, X_audio, Y_genre, Y_niche, Y_joint, seed=SEED, test_size=0.15, val_size=0.15):
    n = len(X_text)
    indices = np.arange(n)

    if HAS_ITERSTRAT:
        msss1 = MultilabelStratifiedShuffleSplit(
            n_splits=1,
            test_size=test_size,
            random_state=seed
        )
        train_val_idx, test_idx = next(msss1.split(indices.reshape(-1, 1), Y_joint))

        Y_joint_train_val = Y_joint[train_val_idx]
        relative_val_size = val_size / (1 - test_size)

        msss2 = MultilabelStratifiedShuffleSplit(
            n_splits=1,
            test_size=relative_val_size,
            random_state=seed
        )
        train_idx_rel, val_idx_rel = next(
            msss2.split(train_val_idx.reshape(-1, 1), Y_joint_train_val)
        )

        train_idx = train_val_idx[train_idx_rel]
        val_idx = train_val_idx[val_idx_rel]
        split_name = 'iterative stratification on genre+niche labels'
    else:
        relative_val_size = val_size / (1 - test_size)
        train_val_idx, test_idx = train_test_split(indices, test_size=test_size, random_state=seed, shuffle=True)
        train_idx, val_idx = train_test_split(train_val_idx, test_size=relative_val_size, random_state=seed, shuffle=True)
        split_name = 'random split fallback'

    return {
        'method': split_name,
        'train_idx': train_idx,
        'val_idx': val_idx,
        'test_idx': test_idx,

        'X_train_text': X_text[train_idx],
        'X_val_text': X_text[val_idx],
        'X_test_text': X_text[test_idx],

        'X_train_audio': X_audio[train_idx],
        'X_val_audio': X_audio[val_idx],
        'X_test_audio': X_audio[test_idx],

        'Y_genre_train': Y_genre[train_idx],
        'Y_genre_val': Y_genre[val_idx],
        'Y_genre_test': Y_genre[test_idx],

        'Y_niche_train': Y_niche[train_idx],
        'Y_niche_val': Y_niche[val_idx],
        'Y_niche_test': Y_niche[test_idx],

        'titles_train': meta_titles[train_idx],
        'artists_train': meta_artists[train_idx],
        'titles_val': meta_titles[val_idx],
        'artists_val': meta_artists[val_idx],
        'titles_test': meta_titles[test_idx],
        'artists_test': meta_artists[test_idx],
        'row_ids_train': train_idx,
        'row_ids_val': val_idx,
        'row_ids_test': test_idx,
    }


split = multilabel_dual_split(X_text, X_audio, Y_genre, Y_niche, Y_joint_for_split)

print('Split method:', split['method'])
print('Train/Val/Test rows:', len(split['train_idx']), len(split['val_idx']), len(split['test_idx']))
print('Genre sizes:', split['Y_genre_train'].shape, split['Y_genre_val'].shape, split['Y_genre_test'].shape)
print('Niche sizes:', split['Y_niche_train'].shape, split['Y_niche_val'].shape, split['Y_niche_test'].shape)

# Final guarantee: all model sections below use these same row IDs.
print('Shared test row IDs sample:', split['row_ids_test'][:10])

Split method: iterative stratification on genre+niche labels
Train/Val/Test rows: 381211 81690 81559
Genre sizes: (381211, 10) (81690, 10) (81559, 10)
Niche sizes: (381211, 754) (81690, 754) (81559, 754)
Shared test row IDs sample: [13 14 15 23 27 32 37 47 55 56]


## 8) Shared evaluation helpers


In [11]:
def evaluate_multilabel(y_true, y_probs, threshold=0.5, model_name='Model', label_type='genre'):
    y_pred = (y_probs >= threshold).astype(int)
    return {
        'label_type': label_type,
        'model': model_name,
        'threshold': float(threshold),
        'micro_f1': f1_score(y_true, y_pred, average='micro', zero_division=0),
        'macro_f1': f1_score(y_true, y_pred, average='macro', zero_division=0),
        'precision_micro': precision_score(y_true, y_pred, average='micro', zero_division=0),
        'recall_micro': recall_score(y_true, y_pred, average='micro', zero_division=0),
        'hamming_loss': hamming_loss(y_true, y_pred),
        'pred_binary': y_pred,
    }


def tune_threshold(y_true, y_probs, thresholds=None):
    if thresholds is None:
        thresholds = np.arange(0.1, 0.71, 0.05)
    rows = []
    for t in thresholds:
        ev = evaluate_multilabel(y_true, y_probs, threshold=t)
        rows.append({
            'threshold': float(t),
            'micro_f1': ev['micro_f1'],
            'macro_f1': ev['macro_f1'],
            'hamming_loss': ev['hamming_loss'],
        })
    return pd.DataFrame(rows).sort_values(['micro_f1', 'macro_f1'], ascending=False)


def labels_from_binary(binary_row, classes):
    return [classes[i] for i, v in enumerate(binary_row) if v == 1]


def top_k_labels(prob_row, classes, k=5):
    top_idx = np.argsort(prob_row)[-k:][::-1]
    return [(classes[i], round(float(prob_row[i]), 4)) for i in top_idx]


def make_result_row(y_true, probs, classes, threshold, top_k=5):
    pred_binary = (probs >= threshold).astype(int)
    return {
        'true_labels': labels_from_binary(y_true, classes),
        'predicted_labels': labels_from_binary(pred_binary, classes),
        'top_probabilities': top_k_labels(probs, classes, k=top_k),
    }


def per_label_report(y_true, y_probs, classes, threshold=0.5):
    y_pred = (y_probs >= threshold).astype(int)
    report = classification_report(
        y_true,
        y_pred,
        target_names=list(classes),
        zero_division=0,
        output_dict=True
    )
    rows = []
    for label in classes:
        rows.append({
            'label': label,
            'precision': report[label]['precision'],
            'recall': report[label]['recall'],
            'f1-score': report[label]['f1-score'],
            'support': report[label]['support'],
        })
    return pd.DataFrame(rows).sort_values('f1-score', ascending=False)


## 9) Count-based text features for AdaBoost and XGBoost

This section creates lightweight count-based lyric features. TF-IDF is not used.

Because the dataset is large, the vectorizer is intentionally limited. AdaBoost uses these features on a sampled training subset, while XGBoost uses the full shared training split. Both models still evaluate on the same validation/test songs.


In [12]:
# Pull labels and text from the ONE shared split.
X_train_text = split['X_train_text']
X_val_text = split['X_val_text']
X_test_text = split['X_test_text']

Y_genre_train = split['Y_genre_train']
Y_genre_val = split['Y_genre_val']
Y_genre_test = split['Y_genre_test']

# Full niche matrices are kept for the multi-head model.
Y_niche_train = split['Y_niche_train']
Y_niche_val = split['Y_niche_val']
Y_niche_test = split['Y_niche_test']

# Practical count-vectorizer settings for a large dataset.
# Increase MAX_FEATURES_COUNT later if training time is acceptable.
MAX_FEATURES_COUNT = 1000

count_vectorizer = CountVectorizer(
    max_features=MAX_FEATURES_COUNT,
    ngram_range=(1, 1),
    min_df=10,
    max_df=0.85,
    stop_words='english',
    binary=True,
)

# Fit ONLY on training lyrics, then transform validation/test lyrics.
# This prevents data leakage and keeps AdaBoost/XGBoost on the exact same split.
X_train_counts = count_vectorizer.fit_transform(X_train_text)
X_val_counts = count_vectorizer.transform(X_val_text)
X_test_counts = count_vectorizer.transform(X_test_text)

print('Count-vector shapes:', X_train_counts.shape, X_val_counts.shape, X_test_counts.shape)
print('AdaBoost/XGBoost test row IDs sample:', split['row_ids_test'][:10])


Count-vector shapes: (381211, 1000) (81690, 1000) (81559, 1000)
AdaBoost/XGBoost test row IDs sample: [13 14 15 23 27 32 37 47 55 56]


## 10) Fast boosted-tree training setup for both genre and niche_genres

This section trains both boosted-tree models on both target spaces, but uses different strategies so the notebook does not run forever.

- **AdaBoost trains on both `genre` and `niche_genres`, but only on a sampled subset of the training rows.** This lets AdaBoost go through all labels without requiring many hours.
- **XGBoost trains on both `genre` and `niche_genres` using the full training split.** XGBoost is more efficient for this large multi-label problem.
- All boosted-tree models still use the same validation/test split, so their results remain comparable song-by-song.


In [13]:
def make_writable_csr(X):
    """
    Fixes read-only sparse matrix issues with sklearn/joblib/AdaBoost.
    """
    if sparse.issparse(X):
        X = X.tocsr(copy=True)
        X.sort_indices()
        X.sum_duplicates()
        X.data = X.data.copy()
        X.indices = X.indices.copy()
        X.indptr = X.indptr.copy()
        return X
    return np.array(X, copy=True)


# Make shared feature matrices safe once.
X_train_counts_safe = make_writable_csr(X_train_counts)
X_val_counts_safe = make_writable_csr(X_val_counts)
X_test_counts_safe = make_writable_csr(X_test_counts)


# AdaBoost is trained on a sample so it can still go through BOTH genre and niche labels.
# Increase this later if runtime is acceptable.
ADABOOST_SAMPLE_SIZE = 50000
ADABOOST_N_ESTIMATORS = 10
ADABOOST_LEARNING_RATE = 0.25

rng = np.random.RandomState(SEED)
train_n = X_train_counts_safe.shape[0]
ada_sample_size = min(ADABOOST_SAMPLE_SIZE, train_n)
ada_sample_idx = rng.choice(np.arange(train_n), size=ada_sample_size, replace=False)

X_train_counts_ada_sample = X_train_counts_safe[ada_sample_idx]
Y_genre_train_ada_sample = Y_genre_train[ada_sample_idx]
Y_niche_train_ada_sample = Y_niche_train[ada_sample_idx]

print('AdaBoost training sample size:', ada_sample_size)
print('AdaBoost will still train against genre labels:', Y_genre_train_ada_sample.shape[1])
print('AdaBoost will still train against niche labels:', Y_niche_train_ada_sample.shape[1])
print('XGBoost full training size:', X_train_counts_safe.shape[0])


def make_adaboost_classifier():
    """Create a fast AdaBoost classifier that works across scikit-learn versions."""
    ada_base = DecisionTreeClassifier(max_depth=1, random_state=SEED)
    try:
        return AdaBoostClassifier(
            estimator=ada_base,
            n_estimators=ADABOOST_N_ESTIMATORS,
            learning_rate=ADABOOST_LEARNING_RATE,
            random_state=SEED,
        )
    except TypeError:
        # Older scikit-learn versions use base_estimator instead of estimator.
        return AdaBoostClassifier(
            base_estimator=ada_base,
            n_estimators=ADABOOST_N_ESTIMATORS,
            learning_rate=ADABOOST_LEARNING_RATE,
            random_state=SEED,
        )


def train_ovr_adaboost(X_train, Y_train):
    """
    Train one AdaBoost binary classifier per label.

    IMPORTANT:
    AdaBoost is intentionally kept single-threaded here because
    sklearn/joblib + sparse matrices can crash or slow down when parallelized.
    """
    model = OneVsRestClassifier(
        make_adaboost_classifier(),
        n_jobs=1,
    )
    model.fit(make_writable_csr(X_train), Y_train)
    return model


def train_ovr_xgboost_fast(X_train, Y_train):
    """Train faster one-vs-rest XGBoost binary classifiers on the full train split."""
    if not HAS_XGBOOST:
        raise ImportError("XGBoost is not installed. Run `pip install xgboost` and rerun.")

    model = OneVsRestClassifier(
        XGBClassifier(
            objective='binary:logistic',
            eval_metric='logloss',
            n_estimators=50,
            max_depth=3,
            learning_rate=0.10,
            subsample=0.80,
            colsample_bytree=0.80,
            random_state=SEED,
            tree_method='hist',
            n_jobs=-1,
        ),
        # Keep outer parallelism at 1 so XGBoost can parallelize internally.
        n_jobs=1,
    )
    model.fit(make_writable_csr(X_train), Y_train)
    return model


trained_models = {}
model_label_spaces = {}

# 1) AdaBoost -> main genre labels on sample.
print('\nTraining AdaBoost for genre labels on sampled training rows...')
trained_models[('genre', 'AdaBoost_sample')] = train_ovr_adaboost(
    X_train_counts_ada_sample,
    Y_genre_train_ada_sample
)
model_label_spaces[('genre', 'AdaBoost_sample')] = {
    'classes': mlb_genre.classes_,
    'Y_val': Y_genre_val,
    'Y_test': Y_genre_test,
}

# 2) XGBoost -> main genre labels on full training split.
if HAS_XGBOOST:
    print('\nTraining XGBoost for genre labels on full training split...')
    trained_models[('genre', 'XGBoost_full')] = train_ovr_xgboost_fast(
        X_train_counts_safe,
        Y_genre_train
    )
    model_label_spaces[('genre', 'XGBoost_full')] = {
        'classes': mlb_genre.classes_,
        'Y_val': Y_genre_val,
        'Y_test': Y_genre_test,
    }
else:
    print('\nSkipping XGBoost genre model because xgboost is not installed.')

# 3) AdaBoost -> all niche_genres labels on sample.
print('\nTraining AdaBoost for ALL niche_genres labels on sampled training rows...')
trained_models[('niche', 'AdaBoost_sample')] = train_ovr_adaboost(
    X_train_counts_ada_sample,
    Y_niche_train_ada_sample
)
model_label_spaces[('niche', 'AdaBoost_sample')] = {
    'classes': mlb_niche.classes_,
    'Y_val': Y_niche_val,
    'Y_test': Y_niche_test,
}

# 4) XGBoost -> all niche_genres labels on full training split.
if HAS_XGBOOST:
    print('\nTraining XGBoost for ALL niche_genres labels on full training split...')
    trained_models[('niche', 'XGBoost_full')] = train_ovr_xgboost_fast(
        X_train_counts_safe,
        Y_niche_train
    )
    model_label_spaces[('niche', 'XGBoost_full')] = {
        'classes': mlb_niche.classes_,
        'Y_val': Y_niche_val,
        'Y_test': Y_niche_test,
    }
else:
    print('\nSkipping XGBoost niche model because xgboost is not installed.')

print('\nFinished boosted-tree training setup.')
print('Shared train row IDs sample:', split['row_ids_train'][:10])
print('AdaBoost sampled train row IDs sample:', split['row_ids_train'][ada_sample_idx[:10]])
print('Shared test row IDs sample:', split['row_ids_test'][:10])


AdaBoost training sample size: 50000
AdaBoost will still train against genre labels: 10
AdaBoost will still train against niche labels: 754
XGBoost full training size: 381211

Training AdaBoost for genre labels on sampled training rows...

Training XGBoost for genre labels on full training split...

Training AdaBoost for ALL niche_genres labels on sampled training rows...

Training XGBoost for ALL niche_genres labels on full training split...

Finished boosted-tree training setup.
Shared train row IDs sample: [ 0  1  2  4  6  7  8  9 10 11]
AdaBoost sampled train row IDs sample: [204095 222289 251753 437403 183932 222300 534188 530417 115504 109419]
Shared test row IDs sample: [13 14 15 23 27 32 37 47 55 56]


## 11) Evaluate separate classifiers and tune thresholds


In [14]:
eval_rows = []
model_outputs = {}

for (label_type, model_name), model in trained_models.items():
    label_info = model_label_spaces[(label_type, model_name)]
    Y_va = label_info['Y_val']
    Y_te = label_info['Y_test']
    classes = label_info['classes']

    val_probs = model.predict_proba(X_val_counts_safe)
    test_probs = model.predict_proba(X_test_counts_safe)

    # Some sklearn wrappers may return list-of-arrays. Normalize to (n_samples, n_labels).
    if isinstance(val_probs, list):
        val_probs = np.vstack([p[:, 1] for p in val_probs]).T
    if isinstance(test_probs, list):
        test_probs = np.vstack([p[:, 1] for p in test_probs]).T

    threshold_table = tune_threshold(Y_va, val_probs)
    best_threshold = float(threshold_table.iloc[0]['threshold'])

    ev = evaluate_multilabel(
        Y_te,
        test_probs,
        threshold=best_threshold,
        model_name=model_name,
        label_type=label_type
    )

    eval_rows.append({k: v for k, v in ev.items() if k != 'pred_binary'})

    model_outputs[(label_type, model_name)] = {
        'val_probs': val_probs,
        'test_probs': test_probs,
        'best_threshold': best_threshold,
        'threshold_table': threshold_table,
        'classes': classes,
        'Y_val': Y_va,
        'Y_test': Y_te,
        'pred_binary': ev['pred_binary'],
    }

separate_model_comparison = pd.DataFrame(eval_rows).sort_values(
    ['label_type', 'micro_f1', 'macro_f1'],
    ascending=[True, False, False]
)

separate_model_comparison


,label_type,model,threshold,micro_f1,macro_f1,precision_micro,recall_micro,hamming_loss
1,genre,XGBoost_full,0.20,0.454727,0.279870,0.416526,0.500644,0.120066
0,genre,AdaBoost_sample,0.35,0.414673,0.189332,0.388110,0.445138,0.125666
3,niche,XGBoost_full,0.10,0.180995,0.069169,0.280789,0.133536,0.004691
2,niche,AdaBoost_sample,0.25,0.119039,0.025152,0.179088,0.089147,0.005122


## 12) Per-label precision/recall tables


In [15]:
# Per-label reports for the boosted-tree models that were actually trained.
for report_key in [
    ('genre', 'AdaBoost_sample'),
    ('genre', 'XGBoost_full'),
    ('niche', 'AdaBoost_sample'),
    ('niche', 'XGBoost_full'),
]:
    if report_key in model_outputs:
        label_type, model_name = report_key
        report = per_label_report(
            model_outputs[report_key]['Y_test'],
            model_outputs[report_key]['test_probs'],
            model_outputs[report_key]['classes'],
            threshold=model_outputs[report_key]['best_threshold']
        )
        print(f'{label_type} per-label report model: {model_name}')
        display(report)


genre per-label report model: AdaBoost_sample


,label,precision,recall,f1-score,support
5,Hip-Hop,0.785566,0.614446,0.689548,6147
9,Rock,0.374068,0.998939,0.544310,29215
4,Folk,0.598303,0.112068,0.188776,7549
7,Pop,0.223178,0.148890,0.178618,10632
8,R&B,0.352113,0.104706,0.161413,4059
2,Country,0.487805,0.069950,0.122355,6862
0,Blues,0.526316,0.004182,0.008299,2391
1,Classical,0.000000,0.000000,0.000000,1803
3,Electronic,0.000000,0.000000,0.000000,10355
6,Jazz,0.000000,0.000000,0.000000,2546


genre per-label report model: XGBoost_full


,label,precision,recall,f1-score,support
5,Hip-Hop,0.730284,0.753213,0.741571,6147
9,Rock,0.397741,0.986137,0.566853,29215
2,Country,0.461199,0.228651,0.305729,6862
8,R&B,0.355509,0.248830,0.292754,4059
3,Electronic,0.383962,0.196523,0.259981,10355
4,Folk,0.685691,0.112995,0.194018,7549
7,Pop,0.276244,0.147761,0.192536,10632
0,Blues,0.430956,0.118779,0.186230,2391
6,Jazz,0.530612,0.020424,0.039334,2546
1,Classical,0.720000,0.009983,0.019694,1803


niche per-label report model: AdaBoost_sample


,label,precision,recall,f1-score,support
183,dancehall,0.692384,0.659134,0.675350,1062
657,soca,0.524568,0.691769,0.596677,571
593,ragga,0.551836,0.475791,0.511000,1074
615,riddim,0.520000,0.487500,0.503226,800
648,singeli,1.000000,0.333333,0.500000,3
...,...,...,...,...,...
285,funk pop,0.000000,0.000000,0.000000,11
286,funk rock,0.000000,0.000000,0.000000,444
287,funkot,0.000000,0.000000,0.000000,8
288,funky house,0.000000,0.000000,0.000000,197


niche per-label report model: XGBoost_full


,label,precision,recall,f1-score,support
183,dancehall,0.675943,0.709040,0.692096,1062
657,soca,0.564276,0.730298,0.636641,571
593,ragga,0.550739,0.520484,0.535184,1074
615,riddim,0.541144,0.485000,0.511536,800
626,rumba congolaise,1.000000,0.333333,0.500000,6
...,...,...,...,...,...
456,mandopop,0.000000,0.000000,0.000000,106
455,mambo,0.000000,0.000000,0.000000,11
454,maluku,0.000000,0.000000,0.000000,1
453,maloya,0.000000,0.000000,0.000000,0


## 13) Single unified multi-head model — full lyrics + audio model predicts genre and all niche_genres

This neural model uses the full `Y_genre` and full `Y_niche` matrices. Unlike the sampled AdaBoost training, the multi-head model learns from the full shared training split and predicts both main genres and all niche genres from the same lyrics + audio representation.


In [16]:
VOCAB_SIZE = 100000
MAX_LEN = 4000
EMBED_DIM = 128
EPOCHS = 10
BATCH_SIZE = 32


def make_tokenized_text(train_text, val_text, test_text, vocab_size=VOCAB_SIZE, max_len=MAX_LEN):
    tokenizer = Tokenizer(num_words=vocab_size, oov_token='<OOV>')
    tokenizer.fit_on_texts(train_text)

    X_train_seq = tokenizer.texts_to_sequences(train_text)
    X_val_seq = tokenizer.texts_to_sequences(val_text)
    X_test_seq = tokenizer.texts_to_sequences(test_text)

    X_train_pad = pad_sequences(X_train_seq, maxlen=max_len, padding='post', truncating='post')
    X_val_pad = pad_sequences(X_val_seq, maxlen=max_len, padding='post', truncating='post')
    X_test_pad = pad_sequences(X_test_seq, maxlen=max_len, padding='post', truncating='post')

    return tokenizer, X_train_pad, X_val_pad, X_test_pad


tokenizer, X_train_pad, X_val_pad, X_test_pad = make_tokenized_text(X_train_text, X_val_text, X_test_text)

X_train_audio = split['X_train_audio']
X_val_audio = split['X_val_audio']
X_test_audio = split['X_test_audio']

audio_scaler = StandardScaler()
X_train_audio_scaled = audio_scaler.fit_transform(X_train_audio)
X_val_audio_scaled = audio_scaler.transform(X_val_audio)
X_test_audio_scaled = audio_scaler.transform(X_test_audio)

print('Text padded shapes:', X_train_pad.shape, X_val_pad.shape, X_test_pad.shape)
print('Audio scaled shapes:', X_train_audio_scaled.shape, X_val_audio_scaled.shape, X_test_audio_scaled.shape)
print('Multi-head model test row IDs sample:', split['row_ids_test'][:10])


Text padded shapes: (381211, 4000) (81690, 4000) (81559, 4000)
Audio scaled shapes: (381211, 15) (81690, 15) (81559, 15)
Multi-head model test row IDs sample: [13 14 15 23 27 32 37 47 55 56]


In [17]:
def make_callbacks():
    return [
        EarlyStopping(monitor='val_loss', patience=2, restore_best_weights=True),
        ReduceLROnPlateau(monitor='val_loss', factor=0.5, patience=1, min_lr=1e-6),
    ]


def build_multihead_hybrid_model(
    num_genre_labels,
    num_niche_labels,
    num_audio_features,
    vocab_size=VOCAB_SIZE,
    embed_dim=EMBED_DIM,
    max_len=MAX_LEN
):
    text_input = Input(shape=(max_len,), name='text_input')
    x = Embedding(vocab_size, embed_dim, input_length=max_len, name='lyrics_embedding')(text_input)

    # This is intentionally not CNN or BiLSTM.
    # GlobalAveragePooling1D creates a compact lyrics representation.
    x = GlobalAveragePooling1D(name='lyrics_global_average_pooling')(x)
    x = Dense(128, activation='relu', kernel_regularizer=l2(1e-4), name='lyrics_dense')(x)
    x = Dropout(0.4, name='lyrics_dropout')(x)

    audio_input = Input(shape=(num_audio_features,), name='audio_input')
    a = Dense(64, activation='relu', name='audio_dense')(audio_input)
    a = BatchNormalization(name='audio_batch_norm')(a)
    a = Dropout(0.3, name='audio_dropout')(a)

    shared = Concatenate(name='lyrics_audio_concat')([x, a])
    shared = Dense(128, activation='relu', kernel_regularizer=l2(1e-4), name='shared_dense')(shared)
    shared = BatchNormalization(name='shared_batch_norm')(shared)
    shared = Dropout(0.4, name='shared_dropout')(shared)

    genre_branch = Dense(64, activation='relu', name='genre_branch_dense')(shared)
    genre_output = Dense(num_genre_labels, activation='sigmoid', name='genre_output')(genre_branch)

    niche_branch = Dense(64, activation='relu', name='niche_branch_dense')(shared)
    niche_output = Dense(num_niche_labels, activation='sigmoid', name='niche_output')(niche_branch)

    model = Model(
        inputs={'text_input': text_input, 'audio_input': audio_input},
        outputs={'genre_output': genre_output, 'niche_output': niche_output}
    )

    model.compile(
        optimizer=Adam(learning_rate=1e-3),
        loss={
            'genre_output': 'binary_crossentropy',
            'niche_output': 'binary_crossentropy',
        },
        loss_weights={
            'genre_output': 1.0,
            'niche_output': 1.0,
        },
        metrics={
            'genre_output': [Precision(name='precision'), Recall(name='recall')],
            'niche_output': [Precision(name='precision'), Recall(name='recall')],
        }
    )

    return model


multihead_model = build_multihead_hybrid_model(
    num_genre_labels=Y_genre_train.shape[1],
    num_niche_labels=Y_niche_train.shape[1],
    num_audio_features=X_train_audio_scaled.shape[1]
)

multihead_model.summary()


Model: "functional"

┏━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)        ┃ Output Shape      ┃    Param # ┃ Connected to      ┃
┡━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━┩
│ text_input          │ (None, 4000)      │          0 │ -                 │
│ (InputLayer)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ lyrics_embedding    │ (None, 4000, 128) │ 12,800,000 │ text_input[0][0]  │
│ (Embedding)         │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ audio_input         │ (None, 15)        │          0 │ -                 │
│ (InputLayer)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ lyrics_global_aver… │ (None, 128)       │          0 │ lyrics_embedding… │
│ (GlobalAveragePool… │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ audio_dense (Dense) │ (None, 64)        │      1,024 │ audio_input[0][0] │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ lyrics_dense        │ (None, 128)       │     16,512 │ lyrics_global_av… │
│ (Dense)             │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ audio_batch_norm    │ (None, 64)        │        256 │ audio_dense[0][0] │
│ (BatchNormalizatio… │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ lyrics_dropout      │ (None, 128)       │          0 │ lyrics_dense[0][… │
│ (Dropout)           │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ audio_dropout       │ (None, 64)        │          0 │ audio_batch_norm… │
│ (Dropout)           │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ lyrics_audio_concat │ (None, 192)       │          0 │ lyrics_dropout[0… │
│ (Concatenate)       │                   │            │ audio_dropout[0]… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ shared_dense        │ (None, 128)       │     24,704 │ lyrics_audio_con… │
│ (Dense)             │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ shared_batch_norm   │ (None, 128)       │        512 │ shared_dense[0][… │
│ (BatchNormalizatio… │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ shared_dropout      │ (None, 128)       │          0 │ shared_batch_nor… │
│ (Dropout)           │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ genre_branch_dense  │ (None, 64)        │      8,256 │ shared_dropout[0… │
│ (Dense)             │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ niche_branch_dense  │ (None, 64)        │      8,256 │ shared_dropout[0… │
│ (Dense)             │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ genre_output        │ (None, 10)        │        650 │ genre_branch_den… │
│ (Dense)             │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ niche_output        │ (None, 754)       │     49,010 │ niche_branch_den… │
│ (Dense)             │                   │            │                 

 Total params: 12,909,180 (49.24 MB)

 Trainable params: 12,908,796 (49.24 MB)

 Non-trainable params: 384 (1.50 KB)

In [18]:
history_multihead = multihead_model.fit(
    {'text_input': X_train_pad, 'audio_input': X_train_audio_scaled},
    {
        'genre_output': Y_genre_train,
        'niche_output': Y_niche_train,
    },
    validation_data=(
        {'text_input': X_val_pad, 'audio_input': X_val_audio_scaled},
        {
            'genre_output': Y_genre_val,
            'niche_output': Y_niche_val,
        }
    ),
    epochs=EPOCHS,
    batch_size=BATCH_SIZE,
    callbacks=make_callbacks(),
    verbose=1,
)

val_probs_multihead = multihead_model.predict(
    {'text_input': X_val_pad, 'audio_input': X_val_audio_scaled},
    verbose=0
)

test_probs_multihead = multihead_model.predict(
    {'text_input': X_test_pad, 'audio_input': X_test_audio_scaled},
    verbose=0
)

# Keras may return a dict or list depending on version.
if isinstance(test_probs_multihead, dict):
    val_probs_multi_genre = val_probs_multihead['genre_output']
    val_probs_multi_niche = val_probs_multihead['niche_output']
    test_probs_multi_genre = test_probs_multihead['genre_output']
    test_probs_multi_niche = test_probs_multihead['niche_output']
else:
    # Outputs were created in the order: genre_output, niche_output.
    val_probs_multi_genre, val_probs_multi_niche = val_probs_multihead
    test_probs_multi_genre, test_probs_multi_niche = test_probs_multihead

multi_genre_threshold_table = tune_threshold(Y_genre_val, val_probs_multi_genre)
BEST_MULTI_GENRE_THRESHOLD = float(multi_genre_threshold_table.iloc[0]['threshold'])

multi_niche_threshold_table = tune_threshold(Y_niche_val, val_probs_multi_niche)
BEST_MULTI_NICHE_THRESHOLD = float(multi_niche_threshold_table.iloc[0]['threshold'])

multi_genre_eval = evaluate_multilabel(
    Y_genre_test,
    test_probs_multi_genre,
    threshold=BEST_MULTI_GENRE_THRESHOLD,
    model_name='Unified multi-head hybrid model',
    label_type='genre'
)

multi_niche_eval = evaluate_multilabel(
    Y_niche_test,
    test_probs_multi_niche,
    threshold=BEST_MULTI_NICHE_THRESHOLD,
    model_name='Unified multi-head hybrid model',
    label_type='niche'
)

pd.DataFrame([
    {k: v for k, v in multi_genre_eval.items() if k != 'pred_binary'},
    {k: v for k, v in multi_niche_eval.items() if k != 'pred_binary'},
])


Epoch 1/10
11913/11913 ━━━━━━━━━━━━━━━━━━━━ 1179s 99ms/step - genre_output_loss: 0.2284 - genre_output_precision: 0.7158 - genre_output_recall: 0.3003 - loss: 0.2527 - niche_output_loss: 0.0204 - niche_output_precision: 0.0218 - niche_output_recall: 0.0023 - val_genre_output_loss: 0.2105 - val_genre_output_precision: 0.7744 - val_genre_output_recall: 0.3266 - val_loss: 0.2305 - val_niche_output_loss: 0.0174 - val_niche_output_precision: 0.5501 - val_niche_output_recall: 0.0052 - learning_rate: 0.0010
Epoch 2/10
11913/11913 ━━━━━━━━━━━━━━━━━━━━ 1236s 104ms/step - genre_output_loss: 0.2197 - genre_output_precision: 0.7436 - genre_output_recall: 0.3135 - loss: 0.2399 - niche_output_loss: 0.0179 - niche_output_precision: 0.4560 - niche_output_recall: 0.0023 - val_genre_output_loss: 0.2107 - val_genre_output_precision: 0.7673 - val_genre_output_recall: 0.3245 - val_loss: 0.2301 - val_niche_output_loss: 0.0172 - val_niche_output_precision: 0.5987 - val_niche_output_recall: 0.0031 - learning_

,label_type,model,threshold,micro_f1,macro_f1,precision_micro,recall_micro,hamming_loss
0,genre,Unified multi-head hybrid model,0.25,0.535683,0.355713,0.540232,0.531211,0.092088
1,niche,Unified multi-head hybrid model,0.10,0.216154,0.054852,0.179012,0.272744,0.007678


In [19]:
# --- SPLIT CONSISTENCY CHECK ---
# These checks make sure AdaBoost, XGBoost, and the multi-head model are all comparing the exact same test songs.
assert X_test_counts_safe.shape[0] == X_test_pad.shape[0] == X_test_audio_scaled.shape[0] == len(split['row_ids_test'])
assert Y_genre_test.shape[0] == Y_niche_test.shape[0] == len(split['row_ids_test'])
print('Confirmed: AdaBoost, XGBoost, and multi-head model all use the same test rows.')
print('Note: XGBoost niche evaluates only the top-label niche subset, while multi-head evaluates the full niche label space.')
print('Shared test row IDs sample:', split['row_ids_test'][:10])


Confirmed: AdaBoost, XGBoost, and multi-head model all use the same test rows.
Note: XGBoost niche evaluates only the top-label niche subset, while multi-head evaluates the full niche label space.
Shared test row IDs sample: [13 14 15 23 27 32 37 47 55 56]


## 14) Compare AdaBoost, XGBoost, and the unified multi-head model on the identical test songs

In [20]:
all_comparison = pd.concat([
    separate_model_comparison,
    pd.DataFrame([
        {k: v for k, v in multi_genre_eval.items() if k != 'pred_binary'},
        {k: v for k, v in multi_niche_eval.items() if k != 'pred_binary'},
    ])
], ignore_index=True).sort_values(
    ['label_type', 'micro_f1', 'macro_f1'],
    ascending=[True, False, False]
)

all_comparison


,label_type,model,threshold,micro_f1,macro_f1,precision_micro,recall_micro,hamming_loss
4,genre,Unified multi-head hybrid model,0.25,0.535683,0.355713,0.540232,0.531211,0.092088
0,genre,XGBoost_full,0.20,0.454727,0.279870,0.416526,0.500644,0.120066
1,genre,AdaBoost_sample,0.35,0.414673,0.189332,0.388110,0.445138,0.125666
5,niche,Unified multi-head hybrid model,0.10,0.216154,0.054852,0.179012,0.272744,0.007678
2,niche,XGBoost_full,0.10,0.180995,0.069169,0.280789,0.133536,0.004691
3,niche,AdaBoost_sample,0.25,0.119039,0.025152,0.179088,0.089147,0.005122


## 15) Show AdaBoost/XGBoost + multi-head genre and niche_genre predictions for the same song

In [21]:
# Pick the boosted-tree models that were actually trained.
BEST_GENRE_SEPARATE_MODEL = 'XGBoost_full' if ('genre', 'XGBoost_full') in model_outputs else ('AdaBoost_sample' if ('genre', 'AdaBoost_sample') in model_outputs else None)
BEST_NICHE_SEPARATE_MODEL = 'XGBoost_full' if ('niche', 'XGBoost_full') in model_outputs else ('AdaBoost_sample' if ('niche', 'AdaBoost_sample') in model_outputs else None)

print('Boosted-tree genre model:', BEST_GENRE_SEPARATE_MODEL)
print('Boosted-tree niche model:', BEST_NICHE_SEPARATE_MODEL)

rows = []

for i in range(min(10, len(split['titles_test']))):
    row = {
        'shared_test_row_id': int(split['row_ids_test'][i]),
        'track_name': split['titles_test'][i],
        'artists': split['artists_test'][i],
    }

    if BEST_GENRE_SEPARATE_MODEL is not None:
        genre_sep_out = model_outputs[('genre', BEST_GENRE_SEPARATE_MODEL)]
        genre_sep = make_result_row(
            genre_sep_out['Y_test'][i],
            genre_sep_out['test_probs'][i],
            genre_sep_out['classes'],
            genre_sep_out['best_threshold'],
            top_k=5
        )
        row.update({
            'true_genres': genre_sep['true_labels'],
            'adaboost_genre_predictions': genre_sep['predicted_labels'],
            'adaboost_genre_top_probs': genre_sep['top_probabilities'],
        })

    if BEST_NICHE_SEPARATE_MODEL is not None:
        niche_sep_out = model_outputs[('niche', BEST_NICHE_SEPARATE_MODEL)]
        niche_sep = make_result_row(
            niche_sep_out['Y_test'][i],
            niche_sep_out['test_probs'][i],
            niche_sep_out['classes'],
            niche_sep_out['best_threshold'],
            top_k=5
        )
        row.update({
            'true_niche_genres_xgb_top_subset': niche_sep['true_labels'],
            'xgboost_niche_top_subset_predictions': niche_sep['predicted_labels'],
            'xgboost_niche_top_subset_probs': niche_sep['top_probabilities'],
        })

    genre_multi = make_result_row(
        Y_genre_test[i],
        test_probs_multi_genre[i],
        mlb_genre.classes_,
        BEST_MULTI_GENRE_THRESHOLD,
        top_k=5
    )

    niche_multi = make_result_row(
        Y_niche_test[i],
        test_probs_multi_niche[i],
        mlb_niche.classes_,
        BEST_MULTI_NICHE_THRESHOLD,
        top_k=5
    )

    row.update({
        'multihead_true_genres': genre_multi['true_labels'],
        'multihead_genre_predictions': genre_multi['predicted_labels'],
        'multihead_genre_top_probs': genre_multi['top_probabilities'],

        'multihead_true_niche_genres_full': niche_multi['true_labels'],
        'multihead_niche_predictions_full': niche_multi['predicted_labels'],
        'multihead_niche_top_probs_full': niche_multi['top_probabilities'],
    })

    rows.append(row)

same_song_results = pd.DataFrame(rows)
same_song_results


Boosted-tree genre model: XGBoost_full
Boosted-tree niche model: XGBoost_full


,shared_test_row_id,track_name,artists,true_genres,adaboost_genre_predictions,adaboost_genre_top_probs,true_niche_genres_xgb_top_subset,xgboost_niche_top_subset_predictions,xgboost_niche_top_subset_probs,multihead_true_genres,multihead_genre_predictions,multihead_genre_top_probs,multihead_true_niche_genres_full,multihead_niche_predictions_full,multihead_niche_top_probs_full
0,13,Doing It All for You,"[""Checkpoint Charlie""]",[Classical],[Rock],"[(Rock, 0.3692), (Pop, 0.168), (R&B, 0.1576), ...","[avant-garde, krautrock]",[],"[(classic rock, 0.0939), (hard rock, 0.0889), ...",[Classical],[Rock],"[(Rock, 0.3451), (Pop, 0.1934), (Electronic, 0...","[avant-garde, krautrock]",[],"[(country, 0.091), (christian rock, 0.0787), (..."
1,14,Mind Control,"[""Tantric""]",[Rock],[Rock],"[(Rock, 0.3801), (Electronic, 0.1747), (Pop, 0...",[post-grunge],[],"[(metal, 0.0578), (classic rock, 0.0403), (pun...",[Rock],[Rock],"[(Rock, 0.7503), (Pop, 0.1446), (Folk, 0.0447)...",[post-grunge],"[emo, melodic hardcore, metalcore, pop punk, p...","[(pop punk, 0.3496), (punk, 0.3488), (skate pu..."
2,15,Castaway,"[""Dog Fashion Disco""]",[Classical],[Rock],"[(Rock, 0.5736), (Folk, 0.14), (Electronic, 0....","[alternative metal, avant-garde, polka]",[metal],"[(metal, 0.1013), (death metal, 0.0929), (goth...",[Classical],[],"[(Pop, 0.2439), (Rock, 0.1479), (R&B, 0.1298),...","[alternative metal, avant-garde, polka]",[],"[(country, 0.0952), (new jack swing, 0.0521), ..."
3,23,Better,"[""Pink Sweat$"", ""KIRBY""]",[Pop],[Rock],"[(Rock, 0.2617), (Electronic, 0.1708), (Pop, 0...",[alternative r&b],[],"[(r&b, 0.0549), (country, 0.0494), (edm, 0.039...",[Pop],[],"[(R&B, 0.2284), (Pop, 0.2039), (Hip-Hop, 0.156...",[alternative r&b],[r&b],"[(r&b, 0.1023), (alternative r&b, 0.0768), (ne..."
4,27,Juanita - 2008 Remaster,"[""Emmylou Harris""]",[Country],"[Country, Rock]","[(Rock, 0.3555), (Country, 0.2038), (Pop, 0.17...","[alt country, americana, bluegrass, classic co...","[classic country, country]","[(country, 0.1277), (classic country, 0.108), ...",[Country],[Rock],"[(Rock, 0.2973), (Pop, 0.2378), (Country, 0.16...","[alt country, americana, bluegrass, classic co...",[],"[(classic country, 0.0849), (yacht rock, 0.079..."
5,32,Brand New Man - with Luke Combs,"[""Brooks & Dunn"", ""Luke Combs""]",[Country],[Country],"[(Country, 0.39), (Rock, 0.1789), (R&B, 0.1013...","[acoustic country, classic country, country, h...","[acoustic country, country]","[(country, 0.3225), (acoustic country, 0.2074)...",[Country],"[Country, Rock]","[(Rock, 0.4112), (Country, 0.3094), (Pop, 0.22...","[acoustic country, classic country, country, h...","[acoustic country, ccm, christian, country, po...","[(country, 0.3165), (pop punk, 0.1142), (ccm, ..."
6,37,Riot Pump,"[""Boo-Yaa T.R.I.B.E.""]",[Hip-Hop],[Hip-Hop],"[(Hip-Hop, 0.7178), (Country, 0.1892), (Rock, ...","[g-funk, rap metal, west coast hip hop]","[country, east coast hip hop, hip hop, old sch...","[(old school hip hop, 0.3648), (country, 0.125...",[Hip-Hop],[Hip-Hop],"[(Hip-Hop, 0.7734), (Folk, 0.094), (R&B, 0.066...","[g-funk, rap metal, west coast hip hop]","[east coast hip hop, g-funk, gangster rap, hip...","[(old school hip hop, 0.3846), (hip hop, 0.283..."
7,47,Equilibrium,"[""Crowbar""]",[Rock],[Rock],"[(Rock, 0.4441), (Electronic, 0.1714), (Pop, 0...","[doom metal, drone metal, groove metal, sludge...",[],"[(metalcore, 0.0644), (metal, 0.0628), (death ...",[Rock],[Rock],"[(Rock, 0.9809), (Pop, 0.0095), (Folk, 0.0053)...","[doom metal, drone metal, groove metal, sludge...","[black metal, death metal, grindcore, groove m...","[(metal, 0.3628), (death metal, 0.2245), (blac..."
8,55,"Swing, Swing","[""The All-American Rejects""]",[Pop],[Rock],"[(Rock, 0.3066), (Pop, 0.1703), (Country, 0.14...","[emo, pop punk]",[],"[(classic country, 0.0699), (jazz, 0.0535), (c...",[Pop],[Rock],"[(Rock, 0.5668), (Electronic, 0.1629), (Pop, 0...","[emo, pop punk]",[],"[(gothic metal, 0.0809), (gothic rock, 0.0786)..."
9,56,Hard To Be Vain,"[

## 18) Save prediction results to CSV


In [24]:
OUTPUT_RESULTS_CSV = 'same_song_genre_and_niche_predictions.csv'
same_song_results.to_csv(OUTPUT_RESULTS_CSV, index=False)
print('Saved:', OUTPUT_RESULTS_CSV)


Saved: same_song_genre_and_niche_predictions.csv


## Notes

This version intentionally removes unrelated baseline/model sections and keeps only XGBoost, AdaBoost, and the lyrics+audio neural classifier.

The boosted-tree models use count-based lyric features. The lyrics+audio classifier uses lyric sequences plus numeric Spotify/audio features.

For fair comparison, all three model families use the same shared train/validation/test split.
